# Предобработка данных для разработчиков популярной игры

- Автор: Шумилин Андрей Иванович
- Дата: 10.05.2026

### Цели и задачи проекта

<font color='#777778'>Цель проекта: помочь команде популярной игры на основе данных подготовить статью об играх для привлечения новой аудитории.\
                      Задачи:<br>
                              1) Познакомиться с данными, внимательно изучив их.\
                              2) Проверить корректность данных.\
                              3) Провести предобработку данных.\
                              4) Получить срез необходимых данных.</font>

### Описание данных

<font color='#777778'>Данные содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:
- `Name` — название игры.
- `Platform` — название платформы.
- `Year of Release` — год выпуска игры.
- `Genre` — жанр игры.
- `NA sales` — продажи в Северной Америке (в млн проданных копий).
- `EU sales` — продажи в Европе (в млн проданных копий).
- `JP sales` — продажи в Японии (в млн проданных копий).
- `Other sales` — продажи в других странах (в млн проданных копий).
- `Critic Score` — оценка критиков (от 0 до 100).
- `User Score` — оценка пользователей (от 0 до 10).
- `Rating` — рейтинг организации ESRB.</font>


### Содержимое проекта
​
<font color='#777778'>Шаги проекта:\
1 1. Загрузка данных и знакомство с ними\
2 2. Проверка ошибок в данных и их предобработка\
3 3. Фильтрация данных\
4 4. Категоризация данных\
5 5. Итоговый вывод</font>
​
---

## 1. Загрузка данных и знакомство с ними

In [1]:
# Импортируем все нужные библиотеки для проекта

import pandas as pd

In [2]:
df_games = pd.read_csv('Desktop/new_games_dataset.csv')

In [3]:
df_games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


Представленные данные занимают более 1.4 MB. Они соответствуют описанию выше. Названия стобцов совпадают с заявленными.

Всего строк в датафрейме 16956, при этом в некоторых данных встречаются пропуски, а именно: поле `Name` содержит 2 пропуска, `Year of Release` - 275 пропусков, `Genre` - 2 пропуска, `Critic Score` - 8714 прпусков (более половины), `User Score` - 6804 пропуска и наконец столбец `Rating` содержит 6871 пропуск.

Для столбца `Year of Release` представлен тип данных `float64`, что не совсем корректно, так как год - целое число, а значит, следует использовать int, к тому же год - небольшое число и объем памяти, предоставленный для хранения значений в этом столбце, можно уменьшить. Что касается столбцов `EU sales` и `JP sales`, для них используется тип `object`, хотя, судя по описанию, хранят поля именно числа в млн, поэтому резонно привести их к типу `float`. То же самое и со столбцом `User Score`.  

Некоторые особенности данных.

1) Вид столбцов прописан не в очень удобном для работы виде: лучше использовать стиль `snake case`.

---

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма

In [4]:
df_games.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

In [5]:
df_games.columns = df_games.columns.str.lower().str.replace(' ', '_')

In [6]:
df_games #проверка

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
16951,Samurai Warriors: Sanada Maru,PS3,2016.0,Action,0.00,0.0,0.01,0.00,NaN,NaN,NaN
16952,LMA Manager 2007,X360,2006.0,Sports,0.00,0.01,0.0,0.00,NaN,NaN,NaN
16953,Haitaka no Psychedelica,PSV,2016.0,Adventure,0.00,0.0,0.01,0.00,NaN,NaN,NaN
16954,Spirits & Spells,GBA,2003.0,Platform,0.01,0.0,0.0,0.00,NaN,NaN,NaN


### 2.2. Типы данных

Ранее было отмечено, в каких столбцах встречаются неверные типы данных. Скорее всего, это связано с неверной интерпретацией содержимого человеком, создавшим датасет. Также стоит подметить, что pandas преобразовал строки с `int` в `float` в столбце `year_of_release`, так как есть пропуски `NaN`

In [7]:
df_games = df_games.copy()
for column in ['eu_sales', 'jp_sales', 'user_score']:
    df_games.loc[:, column] = df_games[column].replace('unknown', float('nan'))
    
df_games = df_games.dropna(subset=['year_of_release'])

df_games['year_of_release'] = df_games['year_of_release'].astype('int16')

df_games['eu_sales'] = pd.to_numeric(df_games['eu_sales'], errors='coerce')
df_games['eu_sales'] = df_games['eu_sales'].astype('float64')

df_games['jp_sales'] = pd.to_numeric(df_games['jp_sales'], errors='coerce')
df_games['jp_sales'] = df_games['jp_sales'].astype('float64')

df_games['user_score'] = pd.to_numeric(df_games['user_score'], errors='coerce')
df_games['user_score'] = df_games['user_score'].astype('float64')

In [8]:
# Проверка
df_games.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16681 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16679 non-null  object 
 1   platform         16681 non-null  object 
 2   year_of_release  16681 non-null  int16  
 3   genre            16679 non-null  object 
 4   na_sales         16681 non-null  float64
 5   eu_sales         16675 non-null  float64
 6   jp_sales         16677 non-null  float64
 7   other_sales      16681 non-null  float64
 8   critic_score     8085 non-null   float64
 9   user_score       7558 non-null   float64
 10  rating           9901 non-null   object 
dtypes: float64(6), int16(1), object(4)
memory usage: 1.4+ MB


### 2.3. Наличие пропусков в данных

In [9]:
len_df_games = len(df_games)
sum_name = df_games['name'].isna().sum()
sum_name_share = sum_name / len_df_games

sum_genre = df_games['genre'].isna().sum()
sum_genre_share = sum_genre / len_df_games

sum_eu_sales = df_games['eu_sales'].isna().sum()
sum_eu_sales_share = sum_eu_sales / len_df_games

sum_jp_sales = df_games['jp_sales'].isna().sum()
sum_jp_sales_share = sum_jp_sales / len_df_games

sum_critic_score = df_games['critic_score'].isna().sum()
sum_critic_score_share = sum_critic_score / len_df_games

sum_user_score = df_games['user_score'].isna().sum()
sum_user_score_share = sum_user_score / len_df_games

sum_rating = df_games['rating'].isna().sum()
sum_rating_share = sum_rating / len_df_games

print(f'Абсолютное значение пропусков в столбце `name`: {sum_name}\nОтносительное: {sum_name_share}\n')

print(f'Абсолютное значение пропусков в столбце `genre`: {sum_genre}\nОтносительное: {sum_genre_share}\n')

print(f'Абсолютное значение пропусков в столбце `eu_sales`: {sum_eu_sales}\nОтносительное: {sum_eu_sales_share}\n')

print(f'Абсолютное значение пропусков в столбце `jp_sales`: {sum_jp_sales}\nОтносительное: {sum_jp_sales_share}\n')

print(f'Абсолютное значение пропусков в столбце `critic_score`: {sum_critic_score}\nОтносительное: {sum_critic_score_share}\n')

print(f'Абсолютное значение пропусков в столбце `user_score`: {sum_user_score}\nОтносительное: {sum_user_score_share}\n')

print(f'Абсолютное значение пропусков в столбце `rating`: {sum_rating}\nОтносительное: {sum_rating_share}\n')

Абсолютное значение пропусков в столбце `name`: 2
Относительное: 0.00011989688867573886

Абсолютное значение пропусков в столбце `genre`: 2
Относительное: 0.00011989688867573886

Абсолютное значение пропусков в столбце `eu_sales`: 6
Относительное: 0.0003596906660272166

Абсолютное значение пропусков в столбце `jp_sales`: 4
Относительное: 0.00023979377735147773

Абсолютное значение пропусков в столбце `critic_score`: 8596
Относительное: 0.5153168275283256

Абсолютное значение пропусков в столбце `user_score`: 9123
Относительное: 0.5469096576943828

Абсолютное значение пропусков в столбце `rating`: 6780
Относительное: 0.40645045261075474



Выше в переменных посчитано количество пропусков для необходимых столбцов в абсолютном и относительном значениях

Пропуски характерны для следующих столбцов: `name` (2 пропуска), `genre` (2 пропуска), `eu_sales` (6 пропусков), `jp_sales` (4 пропуска) `critic_score` (8596 пропусков), `user_score` (9123 пропуска), `rating` (6870 пропусков).

Причины могут быть разными: человеческий фактор, особенность данных, случайные пропуски, не связанные с самими данными.

Можно либо удалить пропуски, если их немного и их удаление не сильно повлияет на расчеты, а можно заменить их релевантным значением, например, медианой или средним.

---

В столбце `year_of_release` уже были удалены строки с пропусками, поэтому итоговое число строк в датафрейме уменьшилоось

В столбце `name` всего 2 пропуска и по нему можно однозначно идентифицировать строку, поэтому можно просто удалить их

In [10]:
df_games = df_games.dropna(subset=['name'])
df_games = df_games.dropna(subset=['genre'])

In [11]:
#В столбце `critic_score` слишком много пропусков, они не случайны, вероятно, у непопулярных игр нет оценок
df_games['critic_score'] = df_games['critic_score'].fillna(-1) #поэтому заменяем на значение-индикатор -1, которого нет в данных (такой оценки не бывает)
df_games['user_score'] = df_games['user_score'].fillna(-1) #аналогично
df_games['rating'] = df_games['rating'].fillna('uknown_rating') #тут тоже ставим значение-индикатор

In [12]:
#Тут заменяем на среднее значение в зависимости от названия платформы и года выхода игры
for column in ['eu_sales', 'jp_sales']:
    mean_value = df_games.groupby(['platform', 'year_of_release'])[column].transform('mean')
    df_games[column] = df_games[column].fillna(mean_value).fillna(df_games[column].mean())
df_games.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16679 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16679 non-null  object 
 1   platform         16679 non-null  object 
 2   year_of_release  16679 non-null  int16  
 3   genre            16679 non-null  object 
 4   na_sales         16679 non-null  float64
 5   eu_sales         16679 non-null  float64
 6   jp_sales         16679 non-null  float64
 7   other_sales      16679 non-null  float64
 8   critic_score     16679 non-null  float64
 9   user_score       16679 non-null  float64
 10  rating           16679 non-null  object 
dtypes: float64(6), int16(1), object(4)
memory usage: 1.4+ MB


### 2.4. Явные и неявные дубликаты в данных

In [13]:
columns = ['name', 'genre', 'platform', 'rating', 'year_of_release']
def unique_values(columns):
    for column in ['platform','year_of_release','genre','rating']:
        print(f'Уникальные значения в столбце {column}:')
        print(df_games[column].sort_values().unique())
        print(f'Всего уникальных значений: {df_games[column].nunique()}')
        print()
unique_values(columns)

Уникальные значения в столбце platform:
['2600' '3DO' '3DS' 'DC' 'DS' 'GB' 'GBA' 'GC' 'GEN' 'GG' 'N64' 'NES' 'NG'
 'PC' 'PCFX' 'PS' 'PS2' 'PS3' 'PS4' 'PSP' 'PSV' 'SAT' 'SCD' 'SNES' 'TG16'
 'WS' 'Wii' 'WiiU' 'X360' 'XB' 'XOne']
Всего уникальных значений: 31

Уникальные значения в столбце year_of_release:
[1980 1981 1982 1983 1984 1985 1986 1987 1988 1989 1990 1991 1992 1993
 1994 1995 1996 1997 1998 1999 2000 2001 2002 2003 2004 2005 2006 2007
 2008 2009 2010 2011 2012 2013 2014 2015 2016]
Всего уникальных значений: 37

Уникальные значения в столбце genre:
['ACTION' 'ADVENTURE' 'Action' 'Adventure' 'FIGHTING' 'Fighting' 'MISC'
 'Misc' 'PLATFORM' 'PUZZLE' 'Platform' 'Puzzle' 'RACING' 'ROLE-PLAYING'
 'Racing' 'Role-Playing' 'SHOOTER' 'SIMULATION' 'SPORTS' 'STRATEGY'
 'Shooter' 'Simulation' 'Sports' 'Strategy']
Всего уникальных значений: 24

Уникальные значения в столбце rating:
['AO' 'E' 'E10+' 'EC' 'K-A' 'M' 'RP' 'T' 'uknown_rating']
Всего уникальных значений: 9



In [14]:
#Как выяснилось на предыдущем шаге, в столбце 'genre' встречаются неявные дубликаты
#Также в столбце с рейтингом встречается рейтинг K-A, который не входит в список рейтингов ESRB, поэтому мы заменим это значение на uknown_rating
count = (df_games['rating'] == 'K-A').sum()
print(f"Записей с рейтингом 'K-A': {count}")

df_games['rating'] = df_games['rating'].replace('K-A', 'unknown_rating')

Записей с рейтингом 'K-A': 3


In [15]:
#Приведение к нижнему регистру жанра игры
df_games['genre'] = df_games['genre'].str.lower()
df_games['rating'] = df_games['rating'].str.upper()

In [16]:
# Проверка явных дубликатов
duplicates = df_games.duplicated().sum()
print(f"Полных дубликатов строк: {duplicates}")

Полных дубликатов строк: 235


In [17]:
df_games = df_games.drop_duplicates()

In [18]:
df_games #Проверка после обработки дубликатов, простой вывод датафрейма

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006,sports,41.36,28.96,3.77,8.45,76.0,8.0,E
1,Super Mario Bros.,NES,1985,platform,29.08,3.58,6.81,0.77,-1.0,-1.0,UKNOWN_RATING
2,Mario Kart Wii,Wii,2008,racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009,sports,15.61,10.93,3.28,2.95,80.0,8.0,E
4,Pokemon Red/Pokemon Blue,GB,1996,role-playing,11.27,8.89,10.22,1.00,-1.0,-1.0,UKNOWN_RATING
...,...,...,...,...,...,...,...,...,...,...,...
16951,Samurai Warriors: Sanada Maru,PS3,2016,action,0.00,0.00,0.01,0.00,-1.0,-1.0,UKNOWN_RATING
16952,LMA Manager 2007,X360,2006,sports,0.00,0.01,0.00,0.00,-1.0,-1.0,UKNOWN_RATING
16953,Haitaka no Psychedelica,PSV,2016,adventure,0.00,0.00,0.01,0.00,-1.0,-1.0,UKNOWN_RATING
16954,Spirits & Spells,GBA,2003,platform,0.01,0.00,0.00,0.00,-1.0,-1.0,UKNOWN_RATING


Всего дубликатов явных было найдено 235 (по всем столбцам), они были удалены с помощью стандартного метода для удаления явных дубликатов в pandas. Перед этим для устранения неявных дубликатов были проанализированны категориальные столбцы и исправлен регистр в столбце genre на нижний, а в столбце rating на верхний.

In [ ]:
#Всего было строк 16956, осталось 16444

In [19]:
#Абсолютное значение удаленных строк
abs_delete = 16956 - len(df_games)
abs_delete

512

In [20]:
#Относительное значение удаленных строк
abs_delete / 16956

0.03019580089643784

Были изменены типы данных на более корректные в контексте самих данных.

Пропуски были обработаны, в столбце `year_of_release` удалены строки с пропущенными годами, так как их мало, а игра в целом непопулярна или года не нашлись. Также и с жанром: важно точно знать жанр игры, поэтому замена на значение-индикатор здесь не подойдет. В некоторых столбцах пропуски были заменены на значения-индикаторы или среднее значение.

Было выявлено 12 неявных дубликатов с жанрами игр, устранены, также удалены полные дубликаты строк.
В процессе предобработки данных удалены 512 строк. что составляет чуть более 3% данных.

## 3. Фильтрация данных

In [21]:
df_actual = df_games[(df_games['year_of_release'] >= 2000) & (df_games['year_of_release'] <= 2013)].copy()

In [22]:
df_actual.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 12781 entries, 0 to 16954
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             12781 non-null  object 
 1   platform         12781 non-null  object 
 2   year_of_release  12781 non-null  int16  
 3   genre            12781 non-null  object 
 4   na_sales         12781 non-null  float64
 5   eu_sales         12781 non-null  float64
 6   jp_sales         12781 non-null  float64
 7   other_sales      12781 non-null  float64
 8   critic_score     12781 non-null  float64
 9   user_score       12781 non-null  float64
 10  rating           12781 non-null  object 
dtypes: float64(6), int16(1), object(4)
memory usage: 1.1+ MB


Всего строк 12781, то есть примерно три четверти данных принадлежат 13 летнему периоду с начала XXI века.

## 4. Категоризация данных

In [23]:
df_actual.loc[:, 'user_score_group'] = pd.cut(
    df_actual['user_score'], 
    bins=[-2, 0, 3, 8, 10], 
    labels=['Нет оценки', 'Низкая оценка', 'Средняя оценка', 'Высокая оценка'],
    right=False
)

In [24]:
# Проверка
df_actual

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,user_score_group
0,Wii Sports,Wii,2006,sports,41.36,28.96,3.77,8.45,76.0,8.0,E,Высокая оценка
2,Mario Kart Wii,Wii,2008,racing,15.68,12.76,3.79,3.29,82.0,8.3,E,Высокая оценка
3,Wii Sports Resort,Wii,2009,sports,15.61,10.93,3.28,2.95,80.0,8.0,E,Высокая оценка
6,New Super Mario Bros.,DS,2006,platform,11.28,9.14,6.50,2.88,89.0,8.5,E,Высокая оценка
7,Wii Play,Wii,2006,misc,13.96,9.18,2.93,2.84,58.0,6.6,E,Средняя оценка
...,...,...,...,...,...,...,...,...,...,...,...,...
16947,Men in Black II: Alien Escape,GC,2003,shooter,0.01,0.00,0.00,0.00,-1.0,-1.0,T,Нет оценки
16949,Woody Woodpecker in Crazy Castle 5,GBA,2002,platform,0.01,0.00,0.00,0.00,-1.0,-1.0,UKNOWN_RATING,Нет оценки
16950,SCORE International Baja 1000: The Official Game,PS2,2008,racing,0.00,0.00,0.00,0.00,-1.0,-1.0,UKNOWN_RATING,Нет оценки
16952,LMA Manager 2007,X360,2006,sports,0.00,0.01,0.00,0.00,-1.0,-1.0,UKNOWN_RATING,Нет оценки


- Разделим все игры по оценкам критиков и выделим такие категории: высокая оценка (от 80 до 100 включительно), средняя оценка (от 30 до 80, не включая правую границу интервала) и низкая оценка (от 0 до 30, не включая правую границу интервала).

In [25]:
df_actual.loc[:, 'critic_score_group'] = pd.cut(
    df_actual['critic_score'], 
    bins=[-2, 0, 30, 80, 100], 
    labels=['Нет оценки', 'Низкая оценка', 'Средняя оценка', 'Высокая оценка'],
    right=False
)

In [26]:
# Проверка
df_actual

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,user_score_group,critic_score_group
0,Wii Sports,Wii,2006,sports,41.36,28.96,3.77,8.45,76.0,8.0,E,Высокая оценка,Средняя оценка
2,Mario Kart Wii,Wii,2008,racing,15.68,12.76,3.79,3.29,82.0,8.3,E,Высокая оценка,Высокая оценка
3,Wii Sports Resort,Wii,2009,sports,15.61,10.93,3.28,2.95,80.0,8.0,E,Высокая оценка,Высокая оценка
6,New Super Mario Bros.,DS,2006,platform,11.28,9.14,6.50,2.88,89.0,8.5,E,Высокая оценка,Высокая оценка
7,Wii Play,Wii,2006,misc,13.96,9.18,2.93,2.84,58.0,6.6,E,Средняя оценка,Средняя оценка
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16947,Men in Black II: Alien Escape,GC,2003,shooter,0.01,0.00,0.00,0.00,-1.0,-1.0,T,Нет оценки,Нет оценки
16949,Woody Woodpecker in Crazy Castle 5,GBA,2002,platform,0.01,0.00,0.00,0.00,-1.0,-1.0,UKNOWN_RATING,Нет оценки,Нет оценки
16950,SCORE International Baja 1000: The Official Game,PS2,2008,racing,0.00,0.00,0.00,0.00,-1.0,-1.0,UKNOWN_RATING,Нет оценки,Нет оценки
16952,LMA Manager 2007,X360,2006,sports,0.00,0.01,0.00,0.00,-1.0,-1.0,UKNOWN_RATING,Нет оценки,Нет оценки


- После категоризации данных проверим результат: сгруппируем данные по выделенным категориям и посчитаем количество игр в каждой категории.

In [27]:
df_actual.groupby('user_score_group')['name'].count() #количество игр в каждой категории по user_score

user_score_group
Нет оценки        6298
Низкая оценка      116
Средняя оценка    4081
Высокая оценка    2286
Name: name, dtype: int64

In [28]:
df_actual.groupby('critic_score_group')['name'].count() #количество игр в каждой категории по critic_score

critic_score_group
Нет оценки        5612
Низкая оценка       55
Средняя оценка    5422
Высокая оценка    1692
Name: name, dtype: int64

Была введена новая категория, про которую не говорили в задании: `Нет оценки`, чтобы избежать наличия пропусков

- Выделим топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [29]:
top_platform = df_actual.groupby('platform')['name'].count() #считаем количество игр для каждой платформы
top_platform

platform
3DS      300
DC        31
DS      2120
GB        27
GBA      811
GC       542
N64       70
PC       766
PS       274
PS2     2127
PS3     1087
PS4       16
PSP     1180
PSV      134
WS         4
Wii     1275
WiiU      74
X360    1121
XB       803
XOne      19
Name: name, dtype: int64

In [30]:
platform_sorted = top_platform.sort_values(ascending=False) #соритируем
platform_sorted

platform
PS2     2127
DS      2120
Wii     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
XB       803
PC       766
GC       542
3DS      300
PS       274
PSV      134
WiiU      74
N64       70
DC        31
GB        27
XOne      19
PS4       16
WS         4
Name: name, dtype: int64

In [31]:
platform_sorted[0:7] #выводим топ-7

platform
PS2     2127
DS      2120
Wii     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: name, dtype: int64

## 5. Итоговый вывод

С помощью библиотеки pandas и инструментов Python была проведена большая работа по предобработке данных об играх для разработчиков популярной игры.
Что было сделано:
- Вначале был выгружен датафрейм и произведено знакомство с данными. Строк было 16956, из которых в процессе обработки данных некоторые были удалены. Датафрейм состоит из 11 столбцов, каждый из которых отражает ключевые характеристики игр с точки зрения продаж и популярности. Некоторые типы данных не соответсвовали данным, содержащимся в конкретных столбцах, поэтому пришлось их заменить на более релевантные. Также названия столбцов были приведены к стилю `snake_case`.
- Затем наступила самая большая часть проекта - обработка пропусков и дубликатов. Пропуски были разной природы и по этой причине метод обработки был выбран тоже разный: удаление строк, замена на значение-индикатор или в зависимости от группы замена на среднее значение. Были выявлены  и устранены неявные дубликаты в столбце `genre`, а значение `K-A` в столбце с рейтингом ESRB заменено на `uknown_rating`. Не обделили вниманием и строго явные дубликаты, которых оказалось в датафрейме 235 штук: они полностью удалены. По итогам этой части работы было удалено чуть более 3% данных, а именно 512 строк.

- Далее наступил не менее важный шаг - фильтрация данных: из исходного датафрейма был сделан `df_actual`, который содержит игры только с 2000 по 2013 год включительно. Стоит заметить, что в результате этого была отсеяна примерно четверть всех строк.
- Последняя содержательная часть проекта - категоризация данных. В датафрейм `df_actual` были добавлены столбцы `user_score_group` и `critic_score_group`, которые содержат категории игр по пользовательским оценкам и оценкам критиков. Столбцы содержат также оценку `Нет оценки` во избежание пропусков. Было посчитано количество игр в каждой категории и игр без оценок в группе пользовательских оценок оказалось примерно половина, а в группе оценок критиков таких игр оказалось меньше половины. Также был получен рейтинг самых популярных платформ по количеству выпущенных на них игр (топ-7). Самой популярной оказалась платформа `PS2`.

В заключение стоит сказать, что цель была достигнута, задачи по предобработке данных - знакомство, приведение типов данных, обработка дубликатов и пропусков, фильтрация данных, категоризация данных - были выполнены.